# Power Plant Energy Prediction

Predict power plant energy output (PE) from temperature (AT), exhaust vacuum (V), pressure (AP), and humidity (RH).

See [README.md](README.md) for data, setup, workflow, and evaluation notes.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("power_plant.csv")

In [ ]:
df.head()

In [ ]:
#AT => Temperature
#V => Vaccum
#AP => Pressure
#RH => Humidity

#PE => Produced Energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop("PE",axis=1)
y = df["PE"]

In [ ]:
y.head()

In [ ]:
#Split out data

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
df.shape

In [ ]:
#Scaling Data 

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
type(X_train_scaled)

In [ ]:
type(y_train)

In [ ]:
y_train.shape

In [ ]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

## Deep Learning

In [ ]:
# Build our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model = nn.Sequential(
        #1st Hidden Layer
        nn.Linear(X_train.shape[1],6),
        nn.ReLU(),

        #2nd Hidden Layer
        nn.Linear(6,6),
        nn.ReLU(),

        #Output Layer
        nn.Linear(6,1)
        )
   

    def forward(self,x):
        return self.model(x)

In [ ]:
import torch.optim as optim

model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
#Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf") 

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # total training loss for 1 epoch
    
    for xb,yb in train_loader:
        #xb = features of 1 batch
        #yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) #forward prop ... predicted outputs for this batch
        loss = criterion(outputs,yb) #compute loss
        loss.backward() #back prop .... compute gradients
        optimizer.step() #params update

        running_loss += loss.item() # loss is tensor => python float value

    epoch_train_loss = running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)

    #Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs,yb)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)

    print(
    f"Epoch {epoch+1}/{epochs} ==> "
    f"Training Loss = {epoch_train_loss:.4f} | "
    f"Validation Loss = {epoch_val_loss:.4f}"
)       

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt") #pt or pth file

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss" : train_losses,
    "Validation Loss" : val_losses
})

plt.figure(figsize=(10,8))
plt.plot(loss_df["Training Loss"],label="Training Loss")
plt.plot(loss_df["Validation Loss"],label="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
#Loading the best model

model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
#Evaluate our Model

model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criterion(train_preds,y_train_tensor)
    test_mse_loss = criterion(test_preds,y_test_tensor)
    
print("Training MSE : ",train_mse_loss.item())
print("Testing MSE : ",test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("R2 Score : ",r2_score(y_test,test_preds)*100)

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(),columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values,columns=["Actual Values"])

pd.concat([predicted_df,actual_df],axis=1)